<div dir="rtl">

## بخش ۰: پیش‌پردازش داده‌ها و کنترل استقلال مشاهدات
**پوشش‌دهنده: الزام ج.۵ (کنترل مراجعه‌های تکراری) و اصلاح پیش‌پردازش فاز اول**

قبل از شروع محاسبات احتمالی، داده‌های بالینی (`UDS`) و تصویربرداری (`MRI`) را ادغام و تمیزکاری می‌کنیم:

* **کنترل مراجعات تکراری (الزام ج.۵):** بررسی ستون `NACCID` نشان می‌دهد که در هر دو فایل دقیقاً ۱۲۳۷ بیمار یکتا وجود دارد و برای هر فرد فقط یک ویزیت ثبت شده است؛ بنابراین شرط استقلال مشاهدات کاملاً برقرار است.
* **اصلاح مقادیر اعشاری و پرت (طبق راهنمای NACC و نظر TA):** به دلیل اجرای الگوریتم جایگذاری (Imputation) روی دیتاست اولیه، برخی کدهای گمشده (مثل `9`، `88` و `888`) با اعداد واقعی میانگین گرفته شده‌اند. در این فاز به جای گرد کردن مقادیر اعشاری در متغیر اسمی `HYPERTEN`، فقط کدهای معتبر `0` (بدون سابقه)، `1` (فعال) و `2` (قدیمی) را نگه داشتیم و مقادیر اعشاری را به عنوان `NaN` حذف کردیم. همچنین مقادیر خارج از بازه مجاز پزشکی در متغیرهای کمی (`NACCBMI > 100`، `BPSYS > 230`، `MOCATOTS > 30` و `EDUC > 36`) فیلتر شدند تا در نهایت **۱۰۲۲ بیمار با داده‌های کامل و معتبر** باقی بمانند.

</div>

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns

uds_df = pd.read_csv('../data/udslong_imputed.csv')
mri_df = pd.read_csv('../data/mriscanclean.csv')

selected_uds = [
    'NACCID',
    'NACCAGE',
    'SEX',
    'EDUC',
    'NACCBMI',
    'BPSYS',
    'HYPERTEN',
    'MOCATOTS',
]
selected_mri = [
    'NACCID',
    'LH_SUPERIORFRONTAL_GVOL',
    'RH_SUPERIORFRONTAL_GVOL',
    'LH_SUPERIORFRONTAL_AVGTH',
    'RH_SUPERIORFRONTAL_AVGTH',
]

df_merged = pd.merge(
    uds_df[selected_uds], mri_df[selected_mri], on='NACCID', how='inner'
)

print(
    f'تعداد کل سطرها: {len(df_merged)} | تعداد بیماران یکتا:'
    f" {df_merged['NACCID'].nunique()}"
)

df_clean = df_merged.copy()
df_clean.loc[df_clean['EDUC'] > 36, 'EDUC'] = np.nan
df_clean.loc[
    (df_clean['NACCBMI'] > 100) | (df_clean['NACCBMI'] < 10), 'NACCBMI'
] = np.nan
df_clean.loc[(df_clean['BPSYS'] > 230) | (df_clean['BPSYS'] < 70), 'BPSYS'] = (
    np.nan
)
df_clean.loc[
    (df_clean['MOCATOTS'] > 30) | (df_clean['MOCATOTS'] < 0), 'MOCATOTS'
] = np.nan
df_clean.loc[~df_clean['HYPERTEN'].isin([0.0, 1.0, 2.0]), 'HYPERTEN'] = np.nan

df_final = df_clean.dropna().reset_index(drop=True)
df_final['HYPERTEN'] = df_final['HYPERTEN'].astype(int)
df_final['SEX'] = df_final['SEX'].astype(int)

print(f'تعداد نهایی بیماران پس از پاکسازی کامل: {len(df_final)}')

تعداد کل سطرها: 1237 | تعداد بیماران یکتا: 1237
تعداد نهایی بیماران پس از پاکسازی کامل: 1022


<div dir="rtl">

## بخش ۱: تعریف رویدادها و متغیرهای تصادفی
**پوشش‌دهنده: تسک ۱ و پیش‌نیاز الزامات ج.۱ و ج.۳**

در این بخش، با توجه به سؤال اصلی پروژه، رویدادهای تصادفی و متغیرهای کمی را روی ۱۰۲۲ بیمار مستقل تعریف می‌کنیم:

**الف) رویدادهای تصادفی (از دو وجه UDS و MRI):**
* **رویداد $A$ (پرفشاری خون فعال - از UDS):** بیمار دارای سابقه فعال یا اخیر فشار خون بالا باشد (`HYPERTEN == 1`).
* **رویداد $B$ (اختلال شناختی - از UDS):** نمره تست شناختی MoCA بیمار کمتر از ۲۶ باشد (`MOCATOTS < 26` که مرز استاندارد پزشکی برای تشخیص افت شناختی است).
* **رویداد $C$ (کاهش حجم قشر پیشانی چپ - از MRI):** حجم ماده خاکستری ناحیه پیشانی فوقانی چپ (`LH_SUPERIORFRONTAL_GVOL`) کمتر از میانه کل بیماران (یعنی کمتر از ۲۱۶۱۶.۵ میلی‌متر مکعب) باشد.

**ب) متغیرهای تصادفی:**
* **متغیر تصادفی گسسته ($X$):** تعداد بیماران دارای اختلال شناختی (وقوع رویداد $B$) در یک نمونه تصادفی $n$ تایی از بیماران.
* **متغیرهای تصادفی پیوسته:** نمره شناختی (`MOCATOTS`) و شاخص توده بدنی (`NACCBMI`) از بخش بالینی، و حجم ماده خاکستری پیشانی فوقانی چپ (`LH_SUPERIORFRONTAL_GVOL`) از بخش تصویربرداری.

</div>

In [2]:
median_lhgvol = df_final['LH_SUPERIORFRONTAL_GVOL'].median()

df_final['Event_A_Hyper'] = df_final['HYPERTEN'] == 1
df_final['Event_B_CogImp'] = df_final['MOCATOTS'] < 26
df_final['Event_C_LowVol'] = df_final['LH_SUPERIORFRONTAL_GVOL'] < median_lhgvol

events_summary = pd.DataFrame({
    'رویداد': [
        'A: پرفشاری خون فعال (UDS)',
        'B: اختلال شناختی MoCA < 26 (UDS)',
        'C: حجم قشر پیشانی زیر میانه (MRI)',
    ],
    'تعداد وقوع': [
        df_final['Event_A_Hyper'].sum(),
        df_final['Event_B_CogImp'].sum(),
        df_final['Event_C_LowVol'].sum(),
    ],
    'احتمال تجربی': [
        df_final['Event_A_Hyper'].mean().round(4),
        df_final['Event_B_CogImp'].mean().round(4),
        df_final['Event_C_LowVol'].mean().round(4),
    ],
})

display(events_summary)

,رویداد,تعداد وقوع,احتمال تجربی
0,A: پرفشاری خون فعال (UDS),396,0.3875
1,B: اختلال شناختی MoCA < 26 (UDS),534,0.5225
2,C: حجم قشر پیشانی زیر میانه (MRI),511,0.5000
